# statcheck-ml version 4: training on the Colab GPU

1. Runtime → Change runtime type → **GPU** (A100).
2. **Run All**. Approve the Google Drive access when it asks: the models are saved in `My Drive/statcheck_v4/`, so a disconnect loses nothing. Run All again and the training continues; a run that finished with the same settings is skipped.
3. The 35 pre-registered runs take about three waves of 12. When the last cell ends, download `My Drive/statcheck_v4/v4_models.tar.gz` into `ml-statcheck/data/` and tell Claude.

In [1]:
# 0. The training kit, embedded, so nothing has to be uploaded. Run this cell; it writes the kit.
import base64, hashlib
KIT_B64 = (
    '...'
)
data = base64.b64decode(KIT_B64)
assert hashlib.sha256(data).hexdigest() == '84c5feebfb0210d3ff06cb6d4a6854743e7a75a88000db9acebe82561d441ab8', 'the embedded kit is damaged'
open('/content/v4_gpu_kit.tar.gz', 'wb').write(data)
print('kit written:', len(data), 'bytes')

kit written: 2853255 bytes


In [2]:
# 1. Mount Google Drive for the results. Approve the access request that opens.
from google.colab import drive
drive.mount('/content/drive')

import os
KIT = '/content/v4_gpu_kit.tar.gz'            # written by cell 0
OUT = '/content/drive/MyDrive/statcheck_v4'   # models and logs are kept here
os.makedirs(OUT, exist_ok=True)
print('kit found:', os.path.exists(KIT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
kit found: True


In [3]:
# 2. Unpack the kit, and keep models/v4 on Drive so the runs survive a disconnect.
%cd /content
!tar xzf "$KIT"
!mkdir -p "$OUT/models_v4"
!rm -rf /content/statcheck-ml/models/v4 && mkdir -p /content/statcheck-ml/models && ln -s "$OUT/models_v4" /content/statcheck-ml/models/v4
%cd /content/statcheck-ml
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content
/content/statcheck-ml
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


In [4]:
# 3. Smoke test: 2 epochs of one run, with a progress table every 20 s. Checks the GPU and times an epoch.
!bash run_v4_gpu.sh smoke

torch 2.11.0+cu130 | CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
gate-old+all: done before with other settings, trains again
start gate-old+all: statcheck_ml.train --device cuda --data dataset/train_v4.jsonl --splits dataset/splits_v4.json --epochs 2 --batch-size 64 --out /content/statcheck-ml/models/v4-smoke/gate-old+all --unit gru --crf --augment 2 --hard-negatives 400 --families font_map,negative_damage,spaced_digits,operator_drop --seed 0
[09:33:16] done 0/1, running 1, waiting 0, elapsed 0 min
    gate-old+all                 epoch  0/2  dev F1 -  0 min
[09:33:36] done 0/1, running 1, waiting 0, elapsed 0 min
    gate-old+all                 epoch  0/2  dev F1 -  0 min
[09:33:56] done 0/1, running 1, waiting 0, elapsed 1 min
    gate-old+all                 epoch  1/2  dev F1 0.845  1 min
[09:34:16] done 0/1, running 1, waiting 0, elapsed 1 min
    gate-old+all                 epoch  1/2  dev F1 0.845  1 min
done  gate-old+all: exit 0 in 75 s
all 1 runs have report.json
2 epoc

In [5]:
# 4. The 35 pre-registered runs (required): 12 train at the same time, the rest wait their turn.
#    A progress table prints every minute. Finished runs on Drive with the same settings are skipped.
!bash run_v4_gpu.sh prereg 12 && cp train_v4.out "$OUT/"

torch 2.11.0+cu130 | CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
gru-softmax-none-s0: done before with other settings, trains again
start gate-none: statcheck_ml.train --device cuda --data dataset/train_v4.jsonl --splits dataset/splits_v4.json --epochs 30 --batch-size 64 --out /content/statcheck-ml/models/v4/gate-none --unit gru --crf --augment 0 --hard-negatives 0 --seed 0
start gate-old: statcheck_ml.train --device cuda --data dataset/train_v4.jsonl --splits dataset/splits_v4.json --epochs 30 --batch-size 64 --out /content/statcheck-ml/models/v4/gate-old --unit gru --crf --augment 2 --hard-negatives 400 --seed 0
start gate-old+font_map: statcheck_ml.train --device cuda --data dataset/train_v4.jsonl --splits dataset/splits_v4.json --epochs 30 --batch-size 64 --out /content/statcheck-ml/models/v4/gate-old+font_map --unit gru --crf --augment 2 --hard-negatives 400 --families font_map --seed 0
start gate-old+negative_damage: statcheck_ml.train --device cuda --data dataset/train_v4.js

In [6]:
# 5. Optional: all 75 runs, exploratory (reported only). The 35 runs already done are skipped.
# !bash run_v4_gpu.sh extended 12 && cp train_v4_extended.out "$OUT/"

In [7]:
# 6. Pack the results into Drive.
!tar czf "$OUT/v4_models.tar.gz" -C /content/statcheck-ml --dereference models/v4 $(ls train_v4*.out 2>/dev/null)
!ls -la "$OUT/v4_models.tar.gz"
print('Download v4_models.tar.gz from My Drive/statcheck_v4 and put it in ml-statcheck/data/.')

-rw------- 1 root root 66490411 Oct  5 10:12 /content/drive/MyDrive/statcheck_v4/v4_models.tar.gz
Download v4_models.tar.gz from My Drive/statcheck_v4 and put it in ml-statcheck/data/.
